# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

I read these findings as evidence to examine constructively rather than claims to "grade." The
paper itself states that its main findings are based mainly on direct portfolio comparisons and
that the study identifies patterns rather than proving cause and effect.

### Finding 1 - Finding #4: The Freshness Multiplier

The paper reports that pages in the 31–90 day freshness window had a 5.43:1
growth-to-decline ratio. It also reports a separate comparison among pages older than one year,
where recently refreshed pages had substantially higher health and impressions than pages last
updated 181–360 days earlier.

**My methodology question:** How was the freshness measurement aligned with the window used
to assign the growth/decline label?

The paper defines trend direction using recent search performance compared with the preceding
period. I would want to confirm that `days_since_last_update` was measured at a point that does
not allow the update event itself to overlap the outcome window being compared. If an update
occurs during the same period used to measure growth, freshness and outcome are contemporaneous
rather than a clean "feature first, outcome later" design.

I would also treat the refreshed-vs-stale comparison as an observed association rather than a
causal refresh effect. Pages selected for refreshing may already differ from untouched pages in
visibility, importance, topic, or previous performance. A stronger follow-up would compare
similar refreshed and non-refreshed pages prospectively.

This does not invalidate the reported pattern; it changes how strongly I would interpret it.

### Finding 2 - Finding #5: Reader Engagement and Search Visibility Move Together

The paper reports substantially higher Health Score for pages with stronger scroll depth and
reader engagement.

**My methodology question:** How much of this relationship remains when the outcome is
independent of the variables being used to define the groups?

The paper defines Health Score partly using scroll depth itself. Therefore, comparing
scroll-depth groups on Health Score can mechanically create part of the observed relationship.
I would want to repeat the comparison using raw outcomes not containing scroll depth — for
example impressions, clicks, or average position — or recompute Health Score without its
scroll component.

If the relationship remains under an independent outcome, that would provide stronger evidence
that engagement is associated with search visibility rather than partly reflecting the
construction of the composite score.

Again, I would interpret this as a methodology question, not as evidence that the paper's
finding is wrong.

In [1]:
# Fresh Colab setup — repo does not persist between sessions
import os

REPO_DIR = "/content/flyrank-ml"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/SuryaK5125/flyrank-ml.git {REPO_DIR}

%cd /content/flyrank-ml

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("Clients:", df["client_id"].nunique())
print("Trend-direction counts:")
print(df["trend_direction"].value_counts(dropna=False))

/content/flyrank-ml
Dataset shape: (30000, 44)
Clients: 32
Trend-direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. My model under an honest split (before/after)

My Week-5 notebook already used a client-grouped 70/30 split, so I am not presenting grouped
validation as a new correction made this week.

Instead, I reconstruct a weaker row-random split as the "before" condition and compare it with
the same client-grouped validation used in Week 5 as the "after" condition.

The purpose is to measure whether allowing pages from the same client into both training and
test sets makes model performance look stronger. Pages belonging to one client can share
unobserved site-level characteristics, so a random row split may reward client-specific
memorisation.

For both validation designs I keep the feature set, preprocessing, model hyperparameters,
random seed, and Precision@50 calculation consistent. I also report ROC-AUC and the test-set
base rate so that differences in Precision@50 are not interpreted without context.

The grouped test asks the more relevant generalisation question for this dataset:
"Can the scoring model rank pages belonging to clients it did not see during training?"

In [2]:
feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
]

model_df = df.dropna(
    subset=feature_cols + ["trend_direction", "client_id"]
).copy()

model_df["label"] = (
    model_df["trend_direction"] == "down"
).astype(int)


def precision_at_k(scores, labels, k=50):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)
    return labels[order[:k]].mean()


def fit_and_evaluate(train_df, test_df, split_name):

    X_train = train_df[feature_cols]
    y_train = train_df["label"]

    X_test = test_df[feature_cols]
    y_test = test_df["label"]

    # Logistic Regression
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    logreg = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    logreg.fit(X_train_scaled, y_train)

    logreg_scores = logreg.predict_proba(
        X_test_scaled
    )[:, 1]

    # Random Forest
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        random_state=42
    )

    rf.fit(X_train, y_train)

    rf_scores = rf.predict_proba(
        X_test
    )[:, 1]

    baseline_scores = test_df["impressions_90d"].values

    overlap = (
        set(train_df["client_id"])
        & set(test_df["client_id"])
    )

    result = {
        "split": split_name,
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "train_clients": train_df["client_id"].nunique(),
        "test_clients": test_df["client_id"].nunique(),
        "overlapping_clients": len(overlap),
        "base_rate": y_test.mean(),

        "impressions_baseline_P@50":
            precision_at_k(
                baseline_scores,
                y_test.values,
                50
            ),

        "logreg_P@50":
            precision_at_k(
                logreg_scores,
                y_test.values,
                50
            ),

        "logreg_AUC":
            roc_auc_score(
                y_test,
                logreg_scores
            ),

        "rf_P@50":
            precision_at_k(
                rf_scores,
                y_test.values,
                50
            ),

        "rf_AUC":
            roc_auc_score(
                y_test,
                rf_scores
            ),
    }

    return result, rf, rf_scores


# BEFORE - random row split
random_train, random_test = train_test_split(
    model_df,
    test_size=0.30,
    random_state=42,
    stratify=model_df["label"]
)

random_result, _, _ = fit_and_evaluate(
    random_train,
    random_test,
    "Random row split"
)


# AFTER - grouped by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

group_train_idx, group_test_idx = next(
    gss.split(
        model_df,
        groups=model_df["client_id"]
    )
)

group_train = model_df.iloc[group_train_idx].copy()
group_test = model_df.iloc[group_test_idx].copy()

group_result, grouped_rf, grouped_rf_scores = fit_and_evaluate(
    group_train,
    group_test,
    "Client-grouped split"
)


comparison = pd.DataFrame([
    random_result,
    group_result
])

pd.set_option("display.max_columns", None)

comparison

,split,train_rows,test_rows,train_clients,test_clients,overlapping_clients,base_rate,impressions_baseline_P@50,logreg_P@50,logreg_AUC,rf_P@50,rf_AUC
0,Random row split,13927,5970,29,28,28,0.600838,0.46,0.70,0.615870,0.94,0.751644
1,Client-grouped split,14114,5783,20,9,0,0.622860,0.52,0.72,0.586066,0.72,0.612471


## 3. Leakage audit and failure examples

I audited the Week-5 model for direct label leakage, identifier leakage, time-window overlap,
and selection effects from missing data.

### Direct leakage

The model does not include `trend_direction`, `trend_pct`, `content_id`, `client_id`, or
existing FlyRank decision fields as model inputs.

As a deliberate leakage test, I added `trend_pct`, which is directly related to the construction
of the target.

The result changed from:

- Honest Random Forest ROC-AUC: **0.612**
- Honest Precision@50: **0.720**

to:

- Leaky ROC-AUC: **1.000**
- Leaky Precision@50: **1.000**

The leaked `trend_pct` feature accounted for approximately **86.8%** of Random Forest feature
importance.

I treat this perfect result as a leakage diagnostic, not as model performance.

### Time-window limitation

The grouped split prevents client overlap, but it does not create temporal separation between
the features and target.

Several model inputs use trailing-90-day measurements, including impressions, clicks, sessions,
CTR, average position, engagement rate, and scroll rate. These overlap the period from which
the current decline proxy is calculated.

Therefore, the present model should not be described as predicting future decline. It is better
interpreted as ranking pages associated with an observed decline state in the current snapshot.

A stronger capstone design would use features from an earlier observation window and measure
decline only in a later outcome window.

### Complete-case limitation

The raw dataset contains **30,000** rows from **32** clients, while the complete-case model
population contains **19,897** rows from **29** clients.

This excludes **10,103 rows, or 33.68% of the dataset**.

The decline rate also changes from **0.5421** in the raw dataset to **0.6008** in the model
population. Therefore, I restrict model-performance claims to the complete-case population
actually evaluated.

### Failure analysis

Because the practical output is a ranked review queue, I inspect two decision-relevant error
types:

1. Non-declining pages that enter the Top-50 queue and consume reviewer capacity.
2. High-impression declining pages that fall outside the Top-50 queue and may therefore be
   missed by a capacity-constrained reviewer.

These examples are used to understand failure modes rather than to redefine the evaluation
metric after seeing the results.

In [3]:
# Leakage audit

forbidden_features = {
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "impressions_prev_30d",
    "content_id",
    "client_id",
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
    "needs_ctr_fix",
    "is_quick_win",
}

print("Forbidden / label-derived fields in honest features:")
print(sorted(set(feature_cols) & forbidden_features))


# Features whose 90-day measurement window overlaps
# the contemporaneous decline-label period
overlapping_window_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
]

print("\n90-day features overlapping the decline-label period:")
print([
    f for f in overlapping_window_features
    if f in feature_cols
])

# Complete-case audit

print("\n--- Complete-case audit ---")

print("Raw rows:", len(df))
print("Model rows:", len(model_df))
print("Rows excluded:", len(df) - len(model_df))

print(
    "Percent excluded:",
    round(
        100 * (1 - len(model_df) / len(df)),
        2
    ),
    "%"
)

print("Raw clients:", df["client_id"].nunique())
print("Model clients:", model_df["client_id"].nunique())

print(
    "Raw decline rate:",
    round(
        df["trend_direction"].eq("down").mean(),
        4
    )
)

print(
    "Complete-case decline rate:",
    round(
        model_df["label"].mean(),
        4
    )
)

# Deliberate leakage test

leaky_df = model_df.copy()
leaky_df["trend_pct_leak"] = (
    leaky_df["trend_pct"].fillna(0)
)

leaky_features = feature_cols + [
    "trend_pct_leak"
]

leaky_train = leaky_df.iloc[
    group_train_idx
].copy()

leaky_test = leaky_df.iloc[
    group_test_idx
].copy()

leaky_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42
)

leaky_rf.fit(
    leaky_train[leaky_features],
    leaky_train["label"]
)

leaky_scores = leaky_rf.predict_proba(
    leaky_test[leaky_features]
)[:, 1]

print("\n--- Deliberate leakage test ---")

print(
    "Honest RF ROC-AUC:",
    round(
        roc_auc_score(
            group_test["label"],
            grouped_rf_scores
        ),
        3
    )
)

print(
    "With trend_pct leak ROC-AUC:",
    round(
        roc_auc_score(
            leaky_test["label"],
            leaky_scores
        ),
        3
    )
)

print(
    "Honest RF Precision@50:",
    round(
        precision_at_k(
            grouped_rf_scores,
            group_test["label"],
            50
        ),
        3
    )
)

print(
    "With trend_pct leak Precision@50:",
    round(
        precision_at_k(
            leaky_scores,
            leaky_test["label"],
            50
        ),
        3
    )
)

leaky_importance = pd.Series(
    leaky_rf.feature_importances_,
    index=leaky_features
).sort_values(ascending=False)

print("\nTop features in deliberately leaky model:")
print(leaky_importance.head(8))

# Real ranking failure examples

audit_test = group_test.copy()

audit_test["rf_score"] = grouped_rf_scores

audit_test = audit_test.sort_values(
    "rf_score",
    ascending=False
).copy()

audit_test["rank"] = np.arange(
    1,
    len(audit_test) + 1
)


# False positives inside Top 50
top50_false_positives = audit_test[
    (audit_test["rank"] <= 50)
    & (audit_test["label"] == 0)
]

print("\n--- Non-declining pages inside Top 50 ---")

display(
    top50_false_positives[
        [
            "content_id",
            "rank",
            "rf_score",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "content_age_days",
            "days_since_last_update",
            "trend_direction",
        ]
    ].head(5)
)


# Important declining pages outside Top 50
missed_decliners = audit_test[
    (audit_test["rank"] > 50)
    & (audit_test["label"] == 1)
].sort_values(
    "impressions_90d",
    ascending=False
)

print("\n--- High-impression declining pages outside Top 50 ---")

display(
    missed_decliners[
        [
            "content_id",
            "rank",
            "rf_score",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "content_age_days",
            "days_since_last_update",
        ]
    ].head(5)
)

Forbidden / label-derived fields in honest features:
[]

90-day features overlapping the decline-label period:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']

--- Complete-case audit ---
Raw rows: 30000
Model rows: 19897
Rows excluded: 10103
Percent excluded: 33.68 %
Raw clients: 32
Model clients: 29
Raw decline rate: 0.5421
Complete-case decline rate: 0.6008

--- Deliberate leakage test ---
Honest RF ROC-AUC: 0.612
With trend_pct leak ROC-AUC: 1.0
Honest RF Precision@50: 0.72
With trend_pct leak Precision@50: 1.0

Top features in deliberately leaky model:
trend_pct_leak            0.867546
impressions_90d           0.043604
avg_position              0.020196
content_age_days          0.017361
clicks_90d                0.010694
days_since_last_update    0.007075
ctr                       0.006857
sessions_90d              0.006729
dtype: float64

--- Non-declining pages inside Top 50 ---


,content_id,rank,rf_score,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,trend_direction
22526,content_1d0963b56227,3,0.836542,3445,3,0.09,39.0,280,104,up
16404,content_cc6b3aef8420,4,0.828225,722,2,0.28,15.0,332,104,up
4050,content_500bd3907331,6,0.822727,4037,4,0.10,5.5,230,104,stable
26547,content_dbb4c75afccc,7,0.816500,636,1,0.16,33.9,300,104,up
5702,content_f8d20d27773f,8,0.811337,2752,22,0.80,4.5,230,104,stable



--- High-impression declining pages outside Top 50 ---


,content_id,rank,rf_score,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update
26844,content_8c19996aa890,4057,0.512132,509252,785,0.15,2.5,445,20
21819,content_4c36c775b818,5154,0.423107,463103,1889,0.41,2.3,445,20
15968,content_66b4046cc144,3679,0.558310,217415,71,0.03,26.6,225,20
11655,content_cea79ef51519,5614,0.342708,208798,490,0.23,5.2,96,20
1448,content_62ed76850efc,5714,0.285053,167858,1272,0.76,5.1,224,20


## 4. Claim rewrite

### Earlier Week-5 claim

> "Both models clearly beat the base rate (+15.6%) and the honest baseline (+38.5%) at
> Precision@50. This is a real, meaningful lift — ranking by a learned combination of signals
> genuinely separates declining pages from stable/growing ones better than either random
> selection or a single-signal rule."

### Revised claim

On a client-grouped holdout containing nine clients not present during training, the Random
Forest measured Precision@50 of **0.720** and ROC-AUC of **0.612**.

The impressions-only baseline measured Precision@50 of **0.520**, while the decline prevalence
in the grouped test population was **0.623**.

These measurements provide directional evidence that combining the available page-level
signals can improve the ordering of a limited content-review queue relative to ranking pages
by impressions alone within the complete-case population evaluated here.

I do not interpret this experiment as evidence that the model predicts future content decline.
The target is a rule-derived proxy for observed recent decline, and several trailing-90-day
features overlap the period used to construct that proxy.

The model is therefore best described as a decision-support ranking model for the observed
snapshot. A stronger predictive claim would require features to be measured before a strictly
later outcome window.

In [4]:
# Metrics supporting the rewritten claim
claim_metrics = pd.DataFrame({
    "Metric": [
        "Grouped RF Precision@50",
        "Grouped RF ROC-AUC",
        "Impressions baseline Precision@50",
        "Grouped test decline rate",
    ],
    "Measured value": [
        group_result["rf_P@50"],
        group_result["rf_AUC"],
        group_result["impressions_baseline_P@50"],
        group_result["base_rate"],
    ],
})

claim_metrics

,Metric,Measured value
0,Grouped RF Precision@50,0.720000
1,Grouped RF ROC-AUC,0.612471
2,Impressions baseline Precision@50,0.520000
3,Grouped test decline rate,0.622860


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.